# Movies Silver Layer

Cleans and standardizes `movies_bronze.movies` into `movies_silver.movies`: deduplication, whitespace trimming, null handling for numerical metrics, type casting, and normalizing the `genres` delimiter. Depends on the **01_bronze_ingestion** notebook.

In [0]:
from pyspark.sql.functions import col, regexp_extract, trim, translate

# 1. Read the raw data from Bronze Delta table
df_bronze = spark.read.table("movies_bronze.movies")

# Optional: Print columns to verify their exact names if needed
print("Original Columns:", df_bronze.columns)

# 2. Apply Silver Layer Cleaning & Transformations
df_silver = (
    df_bronze.dropDuplicates()  # Step A: Remove exact duplicate rows
    # Step B: Trim accidental whitespace
    .withColumn("title", trim(col("title")))
    # Step C: Handle Nulls - fill missing numerical metrics with 0 or default values
    .na.fill({"budget": 0.0, "revenue": 0.0, "vote_count": 0.0})
)

# Step D: Type Casting
df_silver = df_silver \
    .withColumn("budget", col("budget").cast("double")) \
    .withColumn("revenue", col("revenue").cast("double")) \
    .withColumn("vote_count", col("vote_count").cast("float")) \
    .withColumn("popularity_score", col("popularity_score").cast("float")) \
    .withColumn("genres", translate("genres", "|", ","))

# 3. Save the cleaned dataset as the Silver Delta Table
spark.sql("CREATE SCHEMA IF NOT EXISTS movies_silver")

silver_table_name = "movies_silver.movies"

df_silver.write.format("delta").mode("overwrite").saveAsTable(silver_table_name)

print(f"Silver table '{silver_table_name}' successfully created and cleaned!")

# Preview your clean silver data
display(df_silver.limit(10))